# 🔱 Kronos Fine-tune — Binance Perpétuos

Este notebook treina o Kronos em dados reais de perpétuos da Binance.

**Pré-requisitos:**
1. `Runtime → Change runtime type → T4 GPU` (gratuito) ou A100 (Colab Pro)
2. Upload dos CSVs gerados localmente para o Google Drive (pasta `Kronos/data/`)

**Estimativa de tempo (T4 GPU):**
- Tokenizer: ~15 min
- Predictor: ~25 min
- Total: ~40 min por ativo


## 1. Monta Google Drive (salva checkpoints entre sessões)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/Kronos'
os.makedirs(f'{DRIVE_ROOT}/data',      exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/finetuned', exist_ok=True)
print('✅ Drive montado em', DRIVE_ROOT)

## 2. Clona o repositório Kronos

In [ ]:
import os

REPO_DIR = '/content/Kronos'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/shiyu-coder/Kronos.git {REPO_DIR}
else:
    print('Repo já existe, atualizando...')
    !cd {REPO_DIR} && git pull

os.chdir(REPO_DIR)
print('✅ Diretório atual:', os.getcwd())

## 2.5 Instala arquivos de infraestrutura custom

Escreve os módulos `infra/` e `trading/` no repo clonado.
*(Esses arquivos não estão no repo original do Kronos.)*

In [ ]:
# Este célula escreve os arquivos de infraestrutura custom no repo clonado.
# Necessário porque o Colab clona o repo original (sem infra/ e trading/).
import os

CUSTOM_FILES = {
    "infra/__init__.py": """""",
    "infra/binance_data_pipeline.py": """\"\"\"
Camada 1 — Dados (Binance Perpétuos via CCXT)

Fonte recomendada para fine-tune do Kronos:
  - 7+ anos de histórico (vs 6 meses da Hyperliquid)
  - Perpétuos USDM da Binance = mesma série que o oracle da Hyperliquid rastreia
  - Múltiplos ativos: cada símbolo salvo como CSV separado

Instalação:
    pip install ccxt

Uso:
    # Um ativo:
    python3 -m infra.binance_data_pipeline --symbols BTC --interval 1h --days 1000

    # Múltiplos ativos:
    python3 -m infra.binance_data_pipeline --symbols BTC ETH SOL --interval 1h --days 1000

    # Gera configs de fine-tune automaticamente:
    python3 -m infra.binance_data_pipeline --symbols BTC ETH SOL --gen-configs
\"\"\"

import argparse
import time
from pathlib import Path

import pandas as pd

# Binance Futures retorna no máximo 1000 candles por request (CCXT default)
BINANCE_MAX_LIMIT = 1000

# Ativos padrão: perpétuos mais líquidos da Binance (= mais relevantes p/ Hyperliquid)
DEFAULT_SYMBOLS = ["BTC", "ETH", "SOL"]

INTERVAL_MAP = {
    "1m": "1m", "3m": "3m", "5m": "5m", "10m": "10m", "15m": "15m",
    "30m": "30m", "1h": "1h", "4h": "4h", "1d": "1d",
}


# ─────────────────────────────────────────────────────────────────────────────
# Fetch
# ─────────────────────────────────────────────────────────────────────────────

def build_exchange():
    \"\"\"Cria cliente CCXT para Binance Perpétuos USDM (sem autenticação).\"\"\"
    try:
        import ccxt
    except ImportError:
        raise SystemExit("ccxt não instalado. Execute: pip install ccxt")

    exchange = ccxt.binance({
        "options": {"defaultType": "future"},   # USDM perpétuos
        "enableRateLimit": True,                # respeita rate limit automaticamente
    })
    exchange.load_markets()
    return exchange


def fetch_symbol_history(
    exchange,
    symbol: str,
    interval: str,
    days: int,
) -> pd.DataFrame:
    \"\"\"
    Busca `days` dias de candles OHLCV para `symbol` na Binance perpétuos.
    Pagina automaticamente em blocos de 1500 candles.
    \"\"\"
    ccxt_symbol = f"{symbol}/USDT:USDT"   # formato perpétuo CCXT

    # Verifica se o símbolo existe
    if ccxt_symbol not in exchange.markets:
        raise ValueError(f"Símbolo {ccxt_symbol} não encontrado na Binance. "
                         f"Símbolos disponíveis incluem: BTC/USDT:USDT, ETH/USDT:USDT…")

    end_ms   = int(time.time() * 1000)
    since_ms = end_ms - days * 86_400_000
    all_rows = []

    print(f"\\n{'─'*50}")
    print(f"  {symbol}/USDT perp  |  {interval}  |  {days} dias")
    print(f"{'─'*50}")

    while True:
        candles = exchange.fetch_ohlcv(
            ccxt_symbol,
            timeframe=interval,
            since=since_ms,
            limit=BINANCE_MAX_LIMIT,
        )

        if not candles:
            break

        all_rows.extend(candles)
        last_ts = candles[-1][0]
        pct = min((last_ts - (int(time.time() * 1000) - days * 86_400_000)) /
                  (days * 86_400_000) * 100, 100)
        print(f"  {len(all_rows):>7,} candles  [{pct:.0f}%]…", end="\\r")

        # Termina quando o último candle alcança o presente
        # (mais robusto que checar contagem, que varia por exchange)
        if last_ts >= end_ms - 1:
            break

        # Avança cursor para o candle seguinte ao último recebido
        since_ms = last_ts + 1

    print(f"  {len(all_rows):>7,} candles  [100%]   ")

    if not all_rows:
        raise RuntimeError(f"Nenhum dado retornado para {symbol}.")

    df = pd.DataFrame(all_rows, columns=["timestamps", "open", "high", "low", "close", "volume"])
    df["timestamps"] = pd.to_datetime(df["timestamps"], unit="ms")

    # ── Campo obrigatório para CustomKlineDataset do Kronos ──────────────────
    # amount = volume de cotação (USDT movimentado) = base_volume × close_price
    df["amount"] = df["close"] * df["volume"]
    # ─────────────────────────────────────────────────────────────────────────

    df = df.drop_duplicates("timestamps").sort_values("timestamps").reset_index(drop=True)
    print(f"  Período: {df['timestamps'].iloc[0]}  →  {df['timestamps'].iloc[-1]}")
    return df[["timestamps", "open", "high", "low", "close", "volume", "amount"]]


# ─────────────────────────────────────────────────────────────────────────────
# Salvar
# ─────────────────────────────────────────────────────────────────────────────

def save_csv(df: pd.DataFrame, symbol: str, interval: str) -> Path:
    out_dir = Path(__file__).parent.parent / "finetune_csv" / "data"
    out_dir.mkdir(parents=True, exist_ok=True)
    path = out_dir / f"BN_{symbol}_{interval}.csv"
    df.to_csv(path, index=False)
    print(f"  Salvo → {path}  ({len(df):,} linhas)")
    return path


# ─────────────────────────────────────────────────────────────────────────────
# Geração automática de configs de fine-tune
# ─────────────────────────────────────────────────────────────────────────────

CONFIG_TEMPLATE = \"\"\"\\
# Fine-tune Kronos em {symbol}/USDT perpétuo da Binance
# Gerado automaticamente por binance_data_pipeline.py

data:
  data_path: "finetune_csv/data/BN_{symbol}_{interval}.csv"
  lookback_window: 512
  predict_window: 24
  max_context: 512
  clip: 5.0
  train_ratio: 0.85
  val_ratio: 0.10
  test_ratio: 0.05

training:
  tokenizer_epochs: 30
  basemodel_epochs: 20
  batch_size: 32
  log_interval: 50
  num_workers: 4
  seed: 42
  tokenizer_learning_rate: 0.0002
  predictor_learning_rate: 0.000001
  adam_beta1: 0.9
  adam_beta2: 0.95
  adam_weight_decay: 0.1
  accumulation_steps: 2

model_paths:
  pretrained_tokenizer: "NeoQuasar/Kronos-Tokenizer-base"
  pretrained_predictor: "NeoQuasar/Kronos-small"
  exp_name: "BN_{symbol}_{interval}"
  base_path: "finetune_csv/finetuned"
  base_save_path: ""
  finetuned_tokenizer: ""
  tokenizer_save_name: "tokenizer"
  basemodel_save_name: "basemodel"

experiment:
  name: "kronos_bn_{symbol_lower}_{interval}"
  description: "Kronos fine-tuned on Binance {symbol}/USDT perpetual {interval}"
  use_comet: false
  train_tokenizer: true
  train_basemodel: true
  skip_existing: false
  pre_trained_tokenizer: true
  pre_trained_predictor: true

device:
  use_cuda: true
  device_id: 0
\"\"\"


def generate_config(symbol: str, interval: str) -> Path:
    cfg_dir = Path(__file__).parent.parent / "finetune_csv" / "configs"
    cfg_dir.mkdir(parents=True, exist_ok=True)
    path = cfg_dir / f"config_bn_{symbol.lower()}_{interval}.yaml"
    path.write_text(CONFIG_TEMPLATE.format(
        symbol=symbol,
        symbol_lower=symbol.lower(),
        interval=interval,
    ))
    print(f"  Config gerado → {path}")
    return path


# ─────────────────────────────────────────────────────────────────────────────
# Main
# ─────────────────────────────────────────────────────────────────────────────

def main():
    parser = argparse.ArgumentParser(
        description="Baixa histórico de perpétuos Binance para fine-tune do Kronos"
    )
    parser.add_argument("--symbols",    nargs="+", default=DEFAULT_SYMBOLS,
                        metavar="SYM",
                        help=f"Símbolos base (padrão: {' '.join(DEFAULT_SYMBOLS)})")
    parser.add_argument("--interval",   default="1h", choices=list(INTERVAL_MAP),
                        help="Timeframe dos candles (padrão: 1h)")
    parser.add_argument("--days",       type=int, default=1000,
                        help="Dias de histórico (padrão: 1000 ≈ 2.7 anos)")
    parser.add_argument("--gen-configs", action="store_true",
                        help="Gera arquivos YAML de fine-tune automaticamente")
    args = parser.parse_args()

    print("=" * 50)
    print("  Binance Perps → Kronos Dataset Builder")
    print("=" * 50)
    print(f"  Símbolos : {', '.join(args.symbols)}")
    print(f"  Intervalo: {args.interval}")
    print(f"  Histórico: {args.days} dias")
    print()

    exchange = build_exchange()
    saved = []

    for symbol in args.symbols:
        try:
            df = fetch_symbol_history(exchange, symbol, args.interval, args.days)
            path = save_csv(df, symbol, args.interval)
            saved.append((symbol, path))

            if args.gen_configs:
                generate_config(symbol, args.interval)

        except Exception as exc:
            print(f"  ⚠ Erro em {symbol}: {exc}")

    print("\\n" + "=" * 50)
    print("  Concluído!")
    for symbol, path in saved:
        print(f"  ✓ {symbol:6s} → {path.name}")

    if args.gen_configs and saved:
        print()
        print("  Para treinar (um por vez):")
        for symbol, _ in saved:
            print(f"    python3 finetune_csv/train_sequential.py "
                  f"--config finetune_csv/configs/config_bn_{symbol.lower()}_{args.interval}.yaml")
    print("=" * 50)


if __name__ == "__main__":
    main()
""",
    "infra/crypto_data_pipeline.py": """\"\"\"
Camada 1 — Dados

Busca histórico da Hyperliquid e salva em CSV no formato
exato que o finetune_csv/ do Kronos espera:
  timestamps, open, high, low, close, volume

Uso:
    python3 -m infra.crypto_data_pipeline --coin BTC --interval 1h --days 180
\"\"\"

import argparse
import time
from pathlib import Path

import pandas as pd
import requests

INTERVAL_TO_MS = {
    "1m":  60_000,
    "5m":  300_000,
    "15m": 900_000,
    "1h":  3_600_000,
    "4h":  14_400_000,
    "1d":  86_400_000,
}

# Janela segura por batch: 1000 candles evita buracos em ranges históricos
BATCH_CANDLES = 1000
BASE_URL = "https://api.hyperliquid.xyz"


def fetch_batch(coin: str, interval: str, start_ms: int, end_ms: int) -> list:
    payload = {
        "type": "candleSnapshot",
        "req": {"coin": coin, "interval": interval,
                "startTime": start_ms, "endTime": end_ms},
    }
    resp = requests.post(f"{BASE_URL}/info", json=payload, timeout=15)
    resp.raise_for_status()
    return resp.json() or []


def fetch_full_history(coin: str, interval: str, days: int) -> pd.DataFrame:
    \"\"\"
    Busca `days` dias de candles em batches de 1000 e retorna DataFrame.

    Nota: a Hyperliquid tem histórico desde ~Set/2025.
    Para 1h recomenda-se days <= 180.
    \"\"\"
    interval_ms = INTERVAL_TO_MS[interval]
    batch_window_ms = BATCH_CANDLES * interval_ms

    end_ms = int(time.time() * 1000)
    start_ms = end_ms - days * 86_400_000

    all_rows = []
    cursor = start_ms
    empty_batches = 0
    total_batches = (days * 86_400_000) // batch_window_ms + 1

    print(f"Buscando {days} dias de {coin}/{interval} em batches de {BATCH_CANDLES}…")

    while cursor < end_ms:
        batch_end = min(cursor + batch_window_ms, end_ms)
        rows = fetch_batch(coin, interval, cursor, batch_end)

        if not rows:
            # Avança o cursor mesmo em janelas vazias (dados históricos ausentes)
            empty_batches += 1
            cursor = batch_end + interval_ms
            if empty_batches > 20:
                # Muitos batches vazios consecutivos — dados não disponíveis
                print(f"\\nAtenção: {empty_batches} batches vazios consecutivos. "
                      f"Histórico disponível a partir de {all_rows[0]['t'] if all_rows else 'N/A'}.")
                break
            continue

        empty_batches = 0  # reseta contador ao encontrar dados
        all_rows.extend(rows)
        cursor = rows[-1]["t"] + interval_ms
        pct = min((cursor - start_ms) / (end_ms - start_ms) * 100, 100)
        print(f"  {len(all_rows):,} candles  [{pct:.0f}%]…", end="\\r")
        time.sleep(0.15)

    print(f"\\nTotal coletado: {len(all_rows):,} candles")

    if not all_rows:
        raise RuntimeError(
            f"Nenhum dado retornado para {coin}/{interval}. "
            f"Tente reduzir --days (ex: --days 180)."
        )

    df = pd.DataFrame(all_rows)
    df = df.rename(columns={"t": "timestamps", "o": "open", "h": "high",
                             "l": "low",  "c": "close", "v": "volume"})
    df["timestamps"] = pd.to_datetime(df["timestamps"], unit="ms")
    for col in ["open", "high", "low", "close", "volume"]:
        df[col] = pd.to_numeric(df[col])

    df = df.drop_duplicates("timestamps").sort_values("timestamps").reset_index(drop=True)
    print(f"Período: {df['timestamps'].iloc[0]}  →  {df['timestamps'].iloc[-1]}")
    return df[["timestamps", "open", "high", "low", "close", "volume"]]


def save_dataset(df: pd.DataFrame, coin: str, interval: str) -> Path:
    out_dir = Path(__file__).parent.parent / "finetune_csv" / "data"
    out_dir.mkdir(parents=True, exist_ok=True)
    path = out_dir / f"HL_{coin}_{interval}.csv"
    df.to_csv(path, index=False)
    print(f"Salvo: {path}  ({len(df):,} linhas)")
    return path


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--coin",     default="BTC")
    parser.add_argument("--interval", default="1h",
                        choices=list(INTERVAL_TO_MS))
    parser.add_argument("--days",     type=int, default=180,
                        help="Dias de histórico (HL disponível desde ~Set/2025; padrão: 180)")
    args = parser.parse_args()

    df = fetch_full_history(args.coin, args.interval, args.days)
    save_dataset(df, args.coin, args.interval)


if __name__ == "__main__":
    main()
""",
    "infra/build_training_dataset.py": """\"\"\"
Domain-adaptive training dataset builder.

Combines a long Binance history (broad pattern coverage) with a shorter
Hyperliquid history (exact target-exchange distribution) by:
  1. Loading BN_<SYMBOL>_<INTERVAL>.csv  (has `amount` column already)
  2. Loading HL_<SYMBOL>_<INTERVAL>.csv  (usually lacks `amount`)
     For intervals not natively supported by HL (e.g. 10m), a finer
     HL file is resampled: HL_<SYMBOL>_<HL_SRC>.csv → target interval.
  3. Removing Binance rows that overlap with the HL window — HL prices
     are authoritative for that period and must not be duplicated
  4. Computing amount = close * volume for any CSV that lacks it
  5. Concatenating [BN_pre_HL] + [HL] in chronological order
  6. Saving as COMBINED_<SYMBOL>_<INTERVAL>.csv
  7. Auto-generating finetune_csv/configs/config_combined_<symbol>_<interval>.yaml

Usage:
    python3 -m infra.build_training_dataset --symbol BTC --interval 1h
    python3 -m infra.build_training_dataset --symbol BTC --interval 10m
    python3 -m infra.build_training_dataset --symbol ETH --interval 1h
\"\"\"

import argparse
import sys
import textwrap
from pathlib import Path

import pandas as pd
import yaml

REQUIRED_COLS = ["timestamps", "open", "high", "low", "close", "volume", "amount"]

DATA_DIR = Path("finetune_csv/data")
CONFIG_DIR = Path("finetune_csv/configs")

# Hyperliquid native intervals. If the requested interval is not in this set,
# we resample from the nearest finer native interval.
HL_NATIVE = {"1m", "3m", "5m", "15m", "30m", "1h", "4h", "1d"}

# Source interval used when resampling HL data for non-native intervals
HL_RESAMPLE_SRC = {
    "10m": "5m",
    "2m":  "1m",
    "6m":  "3m",
    "20m": "5m",
    "45m": "15m",
}

# predict_window (candles) per interval — how far ahead to forecast
PREDICT_WINDOW = {
    "1m": 30, "3m": 20, "5m": 12, "10m": 6,
    "15m": 8, "30m": 6, "1h": 24, "4h": 12, "1d": 5,
}


def _load(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    df["timestamps"] = pd.to_datetime(df["timestamps"])
    df = df.sort_values("timestamps").reset_index(drop=True)
    if "amount" not in df.columns:
        df["amount"] = df["close"] * df["volume"]
    return df[REQUIRED_COLS]


def _resample_ohlcv(df: pd.DataFrame, rule: str) -> pd.DataFrame:
    \"\"\"Resample a OHLCV+amount DataFrame to a coarser frequency.\"\"\"
    df = df.set_index("timestamps")
    resampled = df.resample(rule).agg({
        "open":   "first",
        "high":   "max",
        "low":    "min",
        "close":  "last",
        "volume": "sum",
        "amount": "sum",
    }).dropna(subset=["open"]).reset_index()
    return resampled[REQUIRED_COLS]


def _load_hl(symbol: str, interval: str) -> pd.DataFrame:
    \"\"\"Load HL data, resampling from a finer interval if needed.\"\"\"
    hl_path = DATA_DIR / f"HL_{symbol}_{interval}.csv"
    if hl_path.exists():
        return _load(hl_path)

    src = HL_RESAMPLE_SRC.get(interval)
    if src is None:
        sys.exit(
            f"HL CSV not found: {hl_path}\\n"
            f"Interval '{interval}' is not natively supported by Hyperliquid "
            f"and has no resample source defined in HL_RESAMPLE_SRC."
        )

    src_path = DATA_DIR / f"HL_{symbol}_{src}.csv"
    if not src_path.exists():
        sys.exit(
            f"HL CSV not found: {hl_path}\\n"
            f"Resample source also missing: {src_path}\\n"
            f"Run first: python3 -m infra.crypto_data_pipeline --coin {symbol} --interval {src}"
        )

    print(f"  HL {interval}: resampling from {src_path.name}…")
    raw = _load(src_path)
    # pandas resample rule: "10min", "5min", etc.
    rule = interval.replace("m", "min").replace("h", "h").replace("d", "D")
    resampled = _resample_ohlcv(raw, rule)
    # Cache the resampled file for future runs
    resampled.to_csv(hl_path, index=False)
    print(f"  HL {interval}: {len(resampled):,} candles (cached → {hl_path.name})")
    return resampled


def build(symbol: str, interval: str) -> Path:
    symbol = symbol.upper()
    bn_path = DATA_DIR / f"BN_{symbol}_{interval}.csv"

    if not bn_path.exists():
        sys.exit(f"Binance CSV not found: {bn_path}")

    bn = _load(bn_path)
    hl = _load_hl(symbol, interval)

    hl_start = hl["timestamps"].min()

    # Binance rows that come strictly BEFORE the HL window
    bn_pre = bn[bn["timestamps"] < hl_start].copy()

    combined = pd.concat([bn_pre, hl], ignore_index=True)
    combined = combined.sort_values("timestamps").reset_index(drop=True)

    out_csv = DATA_DIR / f"COMBINED_{symbol}_{interval}.csv"
    combined.to_csv(out_csv, index=False)

    n_bn = len(bn_pre)
    n_hl = len(hl)
    print(
        f"[build_training_dataset] {symbol}/{interval}\\n"
        f"  BN pre-HL : {n_bn:,} candles  ({bn['timestamps'].min().date()} → {bn_pre['timestamps'].max().date()})\\n"
        f"  HL        : {n_hl:,} candles  ({hl_start.date()} → {hl['timestamps'].max().date()})\\n"
        f"  Combined  : {len(combined):,} candles  → {out_csv}"
    )

    _write_config(symbol, interval, out_csv)
    return out_csv


def _write_config(symbol: str, interval: str, csv_path: Path):
    exp_name = f"COMBINED_{symbol}_{interval}"
    predict_window = PREDICT_WINDOW.get(interval, 24)
    cfg = {
        "data": {
            "data_path": str(csv_path),
            "lookback_window": 512,
            "predict_window": predict_window,
            "max_context": 512,
            "clip": 5.0,
            "train_ratio": 0.85,
            "val_ratio": 0.10,
            "test_ratio": 0.05,
        },
        "training": {
            "tokenizer_epochs": 30,
            "basemodel_epochs": 20,
            "batch_size": 32,
            "log_interval": 50,
            "num_workers": 4,
            "seed": 42,
            "tokenizer_learning_rate": 0.0002,
            "predictor_learning_rate": 0.000001,
            "adam_beta1": 0.9,
            "adam_beta2": 0.95,
            "adam_weight_decay": 0.1,
            "accumulation_steps": 2,
        },
        "model_paths": {
            "pretrained_tokenizer": "NeoQuasar/Kronos-Tokenizer-base",
            "pretrained_predictor": "NeoQuasar/Kronos-small",
            "exp_name": exp_name,
            "base_path": "finetune_csv/finetuned",
            "base_save_path": "",
            "finetuned_tokenizer": "",
            "tokenizer_save_name": "tokenizer",
            "basemodel_save_name": "basemodel",
        },
        "experiment": {
            "name": f"kronos_{exp_name.lower()}",
            "description": (
                f"Kronos fine-tuned on combined Binance+Hyperliquid "
                f"{symbol} {interval} dataset (domain adaptation)"
            ),
            "use_comet": False,
            "train_tokenizer": True,
            "train_basemodel": True,
            "skip_existing": False,
            "pre_trained_tokenizer": True,
            "pre_trained_predictor": True,
        },
        "device": {
            "use_cuda": True,
            "device_id": 0,
        },
    }

    out_yaml = CONFIG_DIR / f"config_combined_{symbol.lower()}_{interval}.yaml"
    CONFIG_DIR.mkdir(parents=True, exist_ok=True)
    with open(out_yaml, "w") as f:
        header = textwrap.dedent(f\"\"\"\\
            # Auto-generated by infra/build_training_dataset.py
            # Strategy: domain adaptation — Binance (long history) + Hyperliquid (target exchange)
            # Binance rows prior to the HL window are prepended for broad pattern coverage;
            # Hyperliquid rows cover the recent period with exact target-exchange prices.
            #
            # To retrain: python3 -m infra.build_training_dataset --symbol {symbol} --interval {interval}
            #             python3 finetune_csv/train_sequential.py --config {out_yaml}
            \"\"\")
        f.write(header + "\\n")
        yaml.dump(cfg, f, default_flow_style=False, allow_unicode=True, indent=2)

    print(f"  Config    : {out_yaml}")


def main():
    parser = argparse.ArgumentParser(
        description="Build a domain-adaptive combined training CSV from Binance + Hyperliquid data."
    )
    parser.add_argument("--symbol",   default="BTC", help="Symbol (BTC, ETH, SOL)")
    parser.add_argument("--interval", default="1h",  help="Candlestick interval (1h, 4h, …)")
    args = parser.parse_args()
    build(args.symbol, args.interval)


if __name__ == "__main__":
    main()
""",
    "infra/ensemble_predictor.py": """\"\"\"
Camada 3 — Validação (parte 2)

EnsemblePredictor: encapsula o KronosPredictor com:
  - Multi-sample: N paths estocásticos → mediana → sinal mais estável
  - Multi-timeframe: confirma sinal em TF maior antes de agir
  - Confidence score calibrado pela concordância entre samples
\"\"\"

import sys
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path(__file__).parent.parent))

from model import Kronos, KronosTokenizer, KronosPredictor
from trading.data_fetcher import fetch_candles, build_kronos_inputs
from trading.signal_generator import generate_signal, Signal, Direction


# Mapeamento de TF menor → TF confirmatório maior
CONFIRMATION_TF = {
    "1m":  "15m",
    "5m":  "1h",
    "15m": "1h",
    "1h":  "4h",
    "4h":  "1d",
    "1d":  "1d",
}


class EnsemblePredictor:
    \"\"\"
    Wrapper sobre KronosPredictor que produz sinais mais confiáveis
    usando múltiplas amostras e confirmação multi-timeframe.
    \"\"\"

    def __init__(
        self,
        model_name: str = "NeoQuasar/Kronos-small",
        tokenizer_name: str = "NeoQuasar/Kronos-Tokenizer-base",
        device: str = "cpu",
        max_context: int = 512,
        sample_count: int = 10,
        temperature: float = 1.0,
        top_p: float = 0.9,
        use_confirmation_tf: bool = True,
        coin: str = "BTC",
        base_url: str = "https://api.hyperliquid.xyz",
    ):
        self.sample_count = sample_count
        self.temperature = temperature
        self.top_p = top_p
        self.use_confirmation_tf = use_confirmation_tf
        self.coin = coin
        self.base_url = base_url

        tokenizer = KronosTokenizer.from_pretrained(tokenizer_name)
        model = Kronos.from_pretrained(model_name)
        self.predictor = KronosPredictor(
            model, tokenizer, device=device, max_context=max_context
        )

    def predict_ensemble(
        self,
        df: pd.DataFrame,
        pred_len: int,
        interval: str,
    ) -> tuple[pd.DataFrame, float]:
        \"\"\"
        Roda `sample_count` forecasts independentes e retorna:
          - forecast_median: mediana dos closes previstos (mais robusto)
          - agreement: fração de samples que concordam na direção (0-1)
        \"\"\"
        x_df, x_ts, y_ts = build_kronos_inputs(df, pred_len, interval)
        current_close = df["close"].iloc[-1]

        all_closes = []
        for _ in range(self.sample_count):
            fc = self.predictor.predict(
                df=x_df, x_timestamp=x_ts, y_timestamp=y_ts,
                pred_len=pred_len, T=self.temperature,
                top_p=self.top_p, sample_count=1,
            )
            all_closes.append(fc["close"].values)

        closes_matrix = np.array(all_closes)          # shape: (sample_count, pred_len)
        median_closes = np.median(closes_matrix, axis=0)

        # Concordância: quantos samples preveem a mesma direção que a mediana?
        final_median = median_closes[-1]
        directions = (closes_matrix[:, -1] > current_close).astype(int)
        majority = 1 if final_median > current_close else 0
        agreement = np.mean(directions == majority)

        # Reconstrói DataFrame com closes medianos
        forecast_median = self.predictor.predict(
            df=x_df, x_timestamp=x_ts, y_timestamp=y_ts,
            pred_len=pred_len, T=self.temperature,
            top_p=self.top_p, sample_count=1,
        ).copy()
        forecast_median["close"] = median_closes

        return forecast_median, float(agreement)

    def get_signal(
        self,
        df: pd.DataFrame,
        pred_len: int,
        interval: str,
        lookback_confirm: int = 100,
    ) -> Signal:
        \"\"\"
        Gera sinal com filtro de concordância e confirmação multi-TF.
        Retorna FLAT se:
          - agreement < 0.6  (samples divergem demais)
          - TF confirmatório aponta direção oposta
        \"\"\"
        forecast_df, agreement = self.predict_ensemble(df, pred_len, interval)

        # Injeta agreement como fator na confidence
        base_signal = generate_signal(df, forecast_df)
        calibrated_confidence = base_signal.confidence * agreement

        # Bloqueia se amostras discordam muito
        if agreement < 0.6:
            return Signal(
                direction=Direction.FLAT,
                confidence=calibrated_confidence,
                entry_price=base_signal.entry_price,
                forecast_horizon=pred_len,
                reason=f"low_agreement={agreement:.2f}",
            )

        if base_signal.direction == Direction.FLAT:
            return base_signal

        # Confirmação multi-timeframe
        if self.use_confirmation_tf:
            confirm_interval = CONFIRMATION_TF.get(interval, interval)
            if confirm_interval != interval:
                confirmed = self._confirm_with_higher_tf(
                    base_signal.direction, confirm_interval, lookback_confirm, pred_len
                )
                if not confirmed:
                    return Signal(
                        direction=Direction.FLAT,
                        confidence=calibrated_confidence,
                        entry_price=base_signal.entry_price,
                        forecast_horizon=pred_len,
                        reason=f"rejected_by_{confirm_interval}_confirmation",
                    )

        return Signal(
            direction=base_signal.direction,
            confidence=calibrated_confidence,
            entry_price=base_signal.entry_price,
            forecast_horizon=pred_len,
            reason=f"{base_signal.reason} | agreement={agreement:.2f}",
        )

    def _confirm_with_higher_tf(
        self,
        direction: Direction,
        confirm_interval: str,
        lookback: int,
        pred_len: int,
    ) -> bool:
        \"\"\"Busca TF maior e verifica se a tendência concorda.\"\"\"
        try:
            df_high = fetch_candles(self.coin, confirm_interval, lookback, self.base_url)
            forecast_high, _ = self.predict_ensemble(df_high, pred_len, confirm_interval)
            signal_high = generate_signal(df_high, forecast_high)

            if signal_high.direction == Direction.FLAT:
                return True  # neutro não bloqueia

            return signal_high.direction == direction
        except Exception:
            return True   # em caso de erro, não bloqueia
""",
    "infra/backtester.py": """\"\"\"
Camada 3 — Validação

Walk-forward backtest: simula o loop de trading no histórico,
candle a candle, sem lookahead. Serve para:
  - Calibrar o threshold de entrada (signal_generator.py)
  - Medir win rate, sharpe, drawdown
  - Decidir se vale ligar ordens reais

Uso:
    python -m infra.backtester --csv finetune_csv/data/HL_BTC_1h.csv \\\\
                                --lookback 200 --pred_len 10 \\\\
                                --threshold 0.015
\"\"\"

import argparse
import sys
from dataclasses import dataclass, field
from pathlib import Path
from typing import List

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path(__file__).parent.parent))

from model import Kronos, KronosTokenizer, KronosPredictor
from trading.signal_generator import generate_signal, Direction


@dataclass
class BacktestResult:
    trades: List[dict] = field(default_factory=list)

    @property
    def n_trades(self): return len(self.trades)

    @property
    def win_rate(self):
        if not self.trades: return 0.0
        wins = sum(1 for t in self.trades if t["pnl_pct"] > 0)
        return wins / len(self.trades)

    @property
    def total_return(self):
        r = 1.0
        for t in self.trades:
            r *= (1 + t["pnl_pct"] / 100)
        return (r - 1) * 100

    @property
    def sharpe(self):
        if len(self.trades) < 2: return 0.0
        rets = [t["pnl_pct"] for t in self.trades]
        return np.mean(rets) / (np.std(rets) + 1e-9) * np.sqrt(252)

    @property
    def max_drawdown(self):
        equity = [1.0]
        for t in self.trades:
            equity.append(equity[-1] * (1 + t["pnl_pct"] / 100))
        peak = equity[0]
        max_dd = 0.0
        for e in equity:
            if e > peak: peak = e
            dd = (peak - e) / peak
            if dd > max_dd: max_dd = dd
        return max_dd * 100

    def summary(self) -> str:
        return (
            f"Trades: {self.n_trades} | "
            f"Win Rate: {self.win_rate:.1%} | "
            f"Total Return: {self.total_return:.2f}% | "
            f"Sharpe: {self.sharpe:.2f} | "
            f"Max DD: {self.max_drawdown:.2f}%"
        )


def run_backtest(
    csv_path: str,
    lookback: int = 200,
    pred_len: int = 10,
    sl_pct: float = 1.5,
    tp_pct: float = 3.0,
    model_name: str = "NeoQuasar/Kronos-small",
    tokenizer_name: str = "NeoQuasar/Kronos-Tokenizer-base",
    device: str = "cpu",
    step: int = 1,              # avança 1 candle por vez (walk-forward)
    max_steps: int = 500,       # limita duração do backtest
) -> BacktestResult:

    df = pd.read_csv(csv_path)
    df["timestamps"] = pd.to_datetime(df["timestamps"])
    df = df.sort_values("timestamps").reset_index(drop=True)
    print(f"Dataset: {len(df)} candles | Walk-forward de {lookback} a {len(df)-pred_len}")

    print("Carregando modelo…")
    tokenizer = KronosTokenizer.from_pretrained(tokenizer_name)
    model = Kronos.from_pretrained(model_name)
    predictor = KronosPredictor(model, tokenizer, device=device, max_context=512)

    result = BacktestResult()
    steps_done = 0

    for i in range(lookback, len(df) - pred_len, step):
        if steps_done >= max_steps:
            break

        hist = df.iloc[i - lookback: i].reset_index(drop=True)
        future = df.iloc[i: i + pred_len].reset_index(drop=True)

        x_df = hist[["open", "high", "low", "close", "volume"]]
        x_ts  = hist["timestamps"]
        y_ts  = future["timestamps"]

        forecast_df = predictor.predict(
            df=x_df, x_timestamp=x_ts, y_timestamp=y_ts,
            pred_len=pred_len, T=1.0, top_p=0.9, sample_count=3,
        )

        signal = generate_signal(hist, forecast_df)

        if signal.direction == Direction.FLAT:
            steps_done += 1
            continue

        entry = hist["close"].iloc[-1]
        is_long = signal.direction == Direction.LONG

        # Simula candle a candle até SL, TP ou fim do horizonte
        # Hyperliquid fee: 0.05% taker on entry + 0.05% taker on exit = 0.10% round-trip
        FEE_RT = 0.10
        pnl_pct = 0.0
        exit_reason = "timeout"
        for _, row in future.iterrows():
            high, low = row["high"], row["low"]
            if is_long:
                if low  <= entry * (1 - sl_pct / 100):
                    pnl_pct = -sl_pct - FEE_RT; exit_reason = "SL"; break
                if high >= entry * (1 + tp_pct / 100):
                    pnl_pct = tp_pct - FEE_RT;  exit_reason = "TP"; break
            else:
                if high >= entry * (1 + sl_pct / 100):
                    pnl_pct = -sl_pct - FEE_RT; exit_reason = "SL"; break
                if low  <= entry * (1 - tp_pct / 100):
                    pnl_pct = tp_pct - FEE_RT;  exit_reason = "TP"; break

        if exit_reason == "timeout":
            exit_price = future["close"].iloc[-1]
            pnl_pct = ((exit_price / entry) - 1) * 100 * (1 if is_long else -1) - FEE_RT

        result.trades.append({
            "timestamp":  str(hist["timestamps"].iloc[-1]),
            "direction":  signal.direction.value,
            "entry":      entry,
            "confidence": signal.confidence,
            "pnl_pct":    round(pnl_pct, 4),
            "exit_reason": exit_reason,
        })

        steps_done += 1
        if steps_done % 20 == 0:
            print(f"  [{steps_done}/{max_steps}] {result.summary()}")

    return result


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--csv",       required=True)
    parser.add_argument("--lookback",  type=int,   default=200)
    parser.add_argument("--pred_len",  type=int,   default=10)
    parser.add_argument("--sl",        type=float, default=1.5)
    parser.add_argument("--tp",        type=float, default=3.0)
    parser.add_argument("--max_steps", type=int,   default=300)
    parser.add_argument("--device",    default="cpu")
    args = parser.parse_args()

    result = run_backtest(
        csv_path=args.csv,
        lookback=args.lookback,
        pred_len=args.pred_len,
        sl_pct=args.sl,
        tp_pct=args.tp,
        max_steps=args.max_steps,
        device=args.device,
    )

    print("\\n" + "=" * 60)
    print("RESULTADO FINAL")
    print("=" * 60)
    print(result.summary())

    out = Path("infra/backtest_results.csv")
    pd.DataFrame(result.trades).to_csv(out, index=False)
    print(f"Trades salvos em: {out}")


if __name__ == "__main__":
    main()
""",
    "infra/state_manager.py": """\"\"\"
Camada 4 — Execução (controle de estado)

Persiste o estado de posições abertas em JSON para que o loop
sobreviva a reinicializações sem criar ordens duplicadas.
\"\"\"

import json
import time
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Optional

STATE_FILE = Path(__file__).parent.parent / "trading" / ".state.json"


@dataclass
class OpenPosition:
    coin: str
    direction: str        # "LONG" | "SHORT"
    entry_price: float
    size_usd: float
    sl_price: float
    tp_price: float
    opened_at: float      # unix timestamp


class StateManager:
    \"\"\"
    Lê/escreve estado de posição em disco.
    Thread-safe para uso em loop único.
    \"\"\"

    def __init__(self, state_file: Path = STATE_FILE):
        self.state_file = state_file
        self.state_file.parent.mkdir(parents=True, exist_ok=True)

    def _load(self) -> dict:
        if not self.state_file.exists():
            return {}
        with open(self.state_file) as f:
            return json.load(f)

    def _save(self, data: dict):
        with open(self.state_file, "w") as f:
            json.dump(data, f, indent=2)

    def get_position(self, coin: str) -> Optional[OpenPosition]:
        data = self._load()
        pos = data.get(coin)
        if pos is None:
            return None
        return OpenPosition(**pos)

    def set_position(self, pos: OpenPosition):
        data = self._load()
        data[pos.coin] = asdict(pos)
        self._save(data)

    def clear_position(self, coin: str):
        data = self._load()
        data.pop(coin, None)
        self._save(data)

    def has_open_position(self, coin: str) -> bool:
        return self.get_position(coin) is not None

    def position_age_seconds(self, coin: str) -> float:
        pos = self.get_position(coin)
        if pos is None:
            return 0.0
        return time.time() - pos.opened_at
""",
    "infra/monitor.py": """\"\"\"
Camada 4 — Execução (alertas)

Envia notificações Telegram a cada evento relevante do workflow.

Setup:
  1. Crie um bot: https://t.me/BotFather → /newbot → copie o token
  2. Abra conversa com o bot e mande /start
  3. Pegue seu chat_id: https://api.telegram.org/bot<TOKEN>/getUpdates
  4. Defina as variáveis de ambiente:
       TELEGRAM_BOT_TOKEN=...
       TELEGRAM_CHAT_ID=...
\"\"\"

import logging
import os
from typing import Optional

import requests

logger = logging.getLogger(__name__)

_BOT_TOKEN = os.getenv("TELEGRAM_BOT_TOKEN", "")
_CHAT_ID   = os.getenv("TELEGRAM_CHAT_ID", "")


def _send(text: str):
    if not _BOT_TOKEN or not _CHAT_ID:
        logger.debug("Telegram não configurado — mensagem suprimida: %s", text)
        return
    try:
        requests.post(
            f"https://api.telegram.org/bot{_BOT_TOKEN}/sendMessage",
            json={"chat_id": _CHAT_ID, "text": text, "parse_mode": "Markdown"},
            timeout=5,
        )
    except Exception as exc:
        logger.warning("Falha ao enviar Telegram: %s", exc)


def alert_signal(coin: str, direction: str, confidence: float, reason: str):
    _send(
        f"📡 *Sinal Kronos*\\n"
        f"Par: `{coin}`\\n"
        f"Direção: `{direction}`\\n"
        f"Confiança: `{confidence:.1%}`\\n"
        f"Motivo: `{reason}`"
    )


def alert_order_placed(coin: str, direction: str, size_usd: float,
                        sl: float, tp: float, dry_run: bool):
    tag = "🔵 DRY RUN" if dry_run else "✅ ORDEM ENVIADA"
    _send(
        f"{tag}\\n"
        f"Par: `{coin}`  |  `{direction}`\\n"
        f"Tamanho: `${size_usd:.2f}`\\n"
        f"SL: `{sl:.2f}`  |  TP: `{tp:.2f}`"
    )


def alert_error(context: str, error: str):
    _send(f"❌ *Erro — {context}*\\n```\\n{error[:400]}\\n```")


def alert_cycle_start(coin: str, interval: str, cycle: int):
    _send(f"🔄 Ciclo #{cycle} | `{coin}/{interval}`")
""",
    "trading/__init__.py": """""",
    "trading/config.py": """from dataclasses import dataclass, field
from typing import Optional


@dataclass
class KronosConfig:
    model_name: str = "NeoQuasar/Kronos-small"
    tokenizer_name: str = "NeoQuasar/Kronos-Tokenizer-base"
    device: str = "cpu"          # "cuda:0" se tiver GPU
    max_context: int = 512
    lookback: int = 200          # candles históricos enviados ao modelo
    pred_len: int = 10           # candles a prever
    temperature: float = 1.0
    top_p: float = 0.9
    sample_count: int = 3        # média de N amostras para reduzir ruído


@dataclass
class HyperliquidConfig:
    # Deixe private_key vazio para modo dry-run (sem ordens reais)
    private_key: str = ""
    account_address: str = ""
    base_url: str = "https://api.hyperliquid.xyz"
    mainnet: bool = True


@dataclass
class TradingConfig:
    coin: str = "BTC"
    interval: str = "1h"         # 1m | 5m | 15m | 1h | 4h | 1d
    leverage: int = 3
    risk_per_trade_pct: float = 1.0   # % do saldo por trade
    sl_pct: float = 1.5               # stop loss em % do preço de entrada
    tp_pct: float = 3.0               # take profit em % do preço de entrada
    dry_run: bool = True              # True = só loga, não envia ordens
    loop_interval_seconds: int = 3600 # frequência do loop principal


@dataclass
class Config:
    kronos: KronosConfig = field(default_factory=KronosConfig)
    hyperliquid: HyperliquidConfig = field(default_factory=HyperliquidConfig)
    trading: TradingConfig = field(default_factory=TradingConfig)
""",
    "trading/data_fetcher.py": """import time
import requests
import pandas as pd
from datetime import datetime, timedelta
from typing import Optional

from .config import TradingConfig

INTERVAL_TO_MS = {
    "1m": 60_000,
    "5m": 300_000,
    "15m": 900_000,
    "1h": 3_600_000,
    "4h": 14_400_000,
    "1d": 86_400_000,
}


def fetch_candles(
    coin: str,
    interval: str,
    lookback: int,
    base_url: str = "https://api.hyperliquid.xyz",
) -> pd.DataFrame:
    \"\"\"
    Busca os últimos `lookback` candles do par `coin` na Hyperliquid.

    Retorna DataFrame com colunas: timestamps, open, high, low, close, volume
    prontas para passar ao KronosPredictor.
    \"\"\"
    interval_ms = INTERVAL_TO_MS.get(interval)
    if interval_ms is None:
        raise ValueError(f"Intervalo '{interval}' inválido. Use: {list(INTERVAL_TO_MS)}")

    end_ms = int(time.time() * 1000)
    start_ms = end_ms - (lookback + 5) * interval_ms  # +5 de margem

    payload = {
        "type": "candleSnapshot",
        "req": {
            "coin": coin,
            "interval": interval,
            "startTime": start_ms,
            "endTime": end_ms,
        },
    }

    resp = requests.post(f"{base_url}/info", json=payload, timeout=10)
    resp.raise_for_status()
    raw = resp.json()

    if not raw:
        raise RuntimeError(f"Hyperliquid retornou resposta vazia para {coin}/{interval}")

    # Formato retornado: lista de dicts com chaves t, o, h, l, c, v, n
    df = pd.DataFrame(raw)
    df = df.rename(columns={"t": "timestamps", "o": "open", "h": "high",
                             "l": "low", "c": "close", "v": "volume"})

    df["timestamps"] = pd.to_datetime(df["timestamps"], unit="ms")
    for col in ["open", "high", "low", "close", "volume"]:
        df[col] = pd.to_numeric(df[col])

    df = df.sort_values("timestamps").reset_index(drop=True)
    df = df.tail(lookback).reset_index(drop=True)

    return df[["timestamps", "open", "high", "low", "close", "volume"]]


def build_kronos_inputs(df: pd.DataFrame, pred_len: int, interval: str):
    \"\"\"
    Divide o DataFrame em (x_df, x_timestamp, y_timestamp) para o KronosPredictor.
    y_timestamp é gerado sinteticamente como futuro imediato.
    \"\"\"
    x_df = df[["open", "high", "low", "close", "volume"]].copy()
    x_timestamp = df["timestamps"].copy()

    interval_ms = INTERVAL_TO_MS[interval]
    last_ts = df["timestamps"].iloc[-1]
    y_timestamps = [
        last_ts + timedelta(milliseconds=interval_ms * (i + 1))
        for i in range(pred_len)
    ]
    y_timestamp = pd.Series(y_timestamps)

    return x_df, x_timestamp, y_timestamp
""",
    "trading/signal_generator.py": """\"\"\"
Signal Generator — converte o forecast do Kronos em direção de trade.

Este é o núcleo estratégico do workflow. A implementação de `generate_signal`
é a decisão mais importante do sistema — veja o TODO abaixo.
\"\"\"

from dataclasses import dataclass
from enum import Enum
from typing import Optional

import pandas as pd


class Direction(str, Enum):
    LONG = "LONG"
    SHORT = "SHORT"
    FLAT = "FLAT"   # sem posição / fechar posição existente


@dataclass
class Signal:
    direction: Direction
    confidence: float          # 0.0 – 1.0
    entry_price: float         # preço atual (último close histórico)
    forecast_horizon: int      # em candles
    reason: str = ""


def generate_signal(
    historical_df: pd.DataFrame,
    forecast_df: pd.DataFrame,
) -> Signal:
    \"\"\"
    Recebe:
      historical_df — DataFrame histórico com [open, high, low, close, volume]
      forecast_df   — DataFrame de previsão do Kronos com as mesmas colunas,
                      indexado pelos timestamps futuros

    Deve retornar um Signal com direction, confidence e entry_price.

    TODO: Implemente sua lógica aqui (5-10 linhas).

    Abordagens possíveis a considerar:

    A) Retorno esperado simples
       - Compara close previsto no horizonte com close atual
       - LONG se (forecast[-1].close / current_close - 1) > threshold
       - Simples, mas sensível a ruído do modelo

    B) Mediana de trajetória
       - Usa vários sample_count e calcula a mediana de todos os forecasts
       - Mais robusto, requer sample_count > 1 no KronosPredictor

    C) Inclinação da tendência (regressão linear)
       - Ajusta uma linha nos closes previstos
       - LONG se inclinação > X%, SHORT se < -X%
       - Filtra melhor ruído de curto prazo

    D) Quebra de nível (high/low previsto vs. atual)
       - LONG se forecast.high.max() > current_close * (1 + threshold)
       - Usa informação dos campos H/L que o Kronos também prevê

    Constraints importantes:
    - confidence muito baixa (<0.4) → retorne FLAT para evitar ruído
    - O Kronos não é calibrado para cripto 24/7 sem fine-tuning — considere
      um threshold conservador inicialmente (ex: >1.5% para agir)
    \"\"\"

    current_close = historical_df["close"].iloc[-1]
    horizon = len(forecast_df)

    # ─── Implemente sua lógica aqui ───────────────────────────────────────────

    predicted_close = forecast_df["close"].iloc[-1]
    expected_return = (predicted_close / current_close) - 1.0
    confidence = min(abs(expected_return) / 0.03, 1.0)  # normaliza em 3%

    if expected_return > 0.015 and confidence > 0.4:
        direction = Direction.LONG
    elif expected_return < -0.015 and confidence > 0.4:
        direction = Direction.SHORT
    else:
        direction = Direction.FLAT

    # ──────────────────────────────────────────────────────────────────────────

    return Signal(
        direction=direction,
        confidence=confidence,
        entry_price=current_close,
        forecast_horizon=horizon,
        reason=f"expected_return={expected_return:.3%}",
    )
""",
    "trading/risk_manager.py": """from dataclasses import dataclass
from typing import Optional

from .config import TradingConfig
from .signal_generator import Direction, Signal


@dataclass
class OrderParams:
    coin: str
    is_buy: bool
    size_usd: float
    sl_price: float
    tp_price: float
    leverage: int


def calculate_order(
    signal: Signal,
    account_balance_usd: float,
    cfg: TradingConfig,
) -> Optional[OrderParams]:
    \"\"\"
    Converte um Signal em parâmetros de ordem.
    Retorna None se o sinal for FLAT ou balanço insuficiente.
    \"\"\"
    if signal.direction == Direction.FLAT:
        return None

    if account_balance_usd < 10:
        return None

    is_buy = signal.direction == Direction.LONG
    risk_usd = account_balance_usd * (cfg.risk_per_trade_pct / 100.0)
    size_usd = risk_usd * cfg.leverage

    if is_buy:
        sl_price = signal.entry_price * (1 - cfg.sl_pct / 100)
        tp_price = signal.entry_price * (1 + cfg.tp_pct / 100)
    else:
        sl_price = signal.entry_price * (1 + cfg.sl_pct / 100)
        tp_price = signal.entry_price * (1 - cfg.tp_pct / 100)

    return OrderParams(
        coin=cfg.coin,
        is_buy=is_buy,
        size_usd=size_usd,
        sl_price=round(sl_price, 2),
        tp_price=round(tp_price, 2),
        leverage=cfg.leverage,
    )
""",
    "trading/order_executor.py": """\"\"\"
Executa ordens na Hyperliquid via hyperliquid-python-sdk.

Instale: pip install hyperliquid-python-sdk
\"\"\"

import logging
from typing import Optional

from .config import HyperliquidConfig, TradingConfig
from .risk_manager import OrderParams

logger = logging.getLogger(__name__)


def get_account_balance(hl_cfg: HyperliquidConfig) -> float:
    \"\"\"Retorna o saldo USDC disponível na conta.\"\"\"
    from hyperliquid.info import Info

    info = Info(hl_cfg.base_url, skip_ws=True)
    state = info.user_state(hl_cfg.account_address)
    return float(state["marginSummary"]["accountValue"])


def place_order(
    params: OrderParams,
    hl_cfg: HyperliquidConfig,
    trading_cfg: TradingConfig,
) -> Optional[dict]:
    \"\"\"
    Envia uma ordem market + SL/TP para a Hyperliquid.

    dry_run=True → apenas loga, sem enviar.
    \"\"\"
    if trading_cfg.dry_run:
        logger.info(
            "[DRY RUN] %s %s | size=$%.2f | SL=%.2f | TP=%.2f | lev=%dx",
            "LONG" if params.is_buy else "SHORT",
            params.coin,
            params.size_usd,
            params.sl_price,
            params.tp_price,
            params.leverage,
        )
        return {"status": "dry_run", "params": params}

    from hyperliquid.exchange import Exchange
    from hyperliquid.utils import constants
    import eth_account

    account = eth_account.Account.from_key(hl_cfg.private_key)
    exchange = Exchange(
        account,
        constants.MAINNET_API_URL if hl_cfg.mainnet else constants.TESTNET_API_URL,
    )

    # Ajusta alavancagem
    exchange.update_leverage(params.leverage, params.coin, is_cross=True)

    # Tamanho em moeda base (ex: BTC)
    # Precisamos do preço atual para converter USD → qty
    from hyperliquid.info import Info
    info = Info(hl_cfg.base_url, skip_ws=True)
    mid_price = float(info.all_mids()[params.coin])
    qty = round(params.size_usd / mid_price, 5)

    # Ordem market com SL/TP
    order_result = exchange.market_open(
        params.coin,
        params.is_buy,
        qty,
        slippage=0.01,
    )
    logger.info("Ordem enviada: %s", order_result)

    # SL via TP/SL order (trigger order)
    sl_order = exchange.order(
        params.coin,
        not params.is_buy,   # lado inverso
        qty,
        params.sl_price,
        {"trigger": {"triggerPx": params.sl_price, "isMarket": True, "tpsl": "sl"}},
        reduce_only=True,
    )
    logger.info("Stop Loss configurado: %s", sl_order)

    tp_order = exchange.order(
        params.coin,
        not params.is_buy,
        qty,
        params.tp_price,
        {"trigger": {"triggerPx": params.tp_price, "isMarket": True, "tpsl": "tp"}},
        reduce_only=True,
    )
    logger.info("Take Profit configurado: %s", tp_order)

    return order_result
""",
    "trading/workflow.py": """\"\"\"
Loop principal — versão completa com todas as camadas de infraestrutura.

Uso:
    python -m trading.workflow
\"\"\"

import logging
import sys
import time
import traceback
from pathlib import Path

sys.path.insert(0, str(Path(__file__).parent.parent))

from trading.config import Config
from trading.data_fetcher import fetch_candles
from trading.signal_generator import Direction
from trading.risk_manager import calculate_order, OrderParams
from trading.order_executor import get_account_balance, place_order
from infra.ensemble_predictor import EnsemblePredictor
from infra.state_manager import StateManager, OpenPosition
from infra import monitor

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger(__name__)


def build_ensemble(cfg: Config) -> EnsemblePredictor:
    logger.info("Carregando EnsemblePredictor (%s)…", cfg.kronos.model_name)
    return EnsemblePredictor(
        model_name=cfg.kronos.model_name,
        tokenizer_name=cfg.kronos.tokenizer_name,
        device=cfg.kronos.device,
        max_context=cfg.kronos.max_context,
        sample_count=cfg.kronos.sample_count,
        temperature=cfg.kronos.temperature,
        top_p=cfg.kronos.top_p,
        use_confirmation_tf=True,
        coin=cfg.trading.coin,
        base_url=cfg.hyperliquid.base_url,
    )


def run_cycle(
    ensemble: EnsemblePredictor,
    state: StateManager,
    cfg: Config,
    cycle: int,
):
    tcfg = cfg.trading
    kcfg = cfg.kronos

    monitor.alert_cycle_start(tcfg.coin, tcfg.interval, cycle)

    # ── 1. Evita abrir nova posição se já há uma aberta ──────────────────────
    if state.has_open_position(tcfg.coin):
        age = state.position_age_seconds(tcfg.coin)
        pos = state.get_position(tcfg.coin)
        logger.info("Posição %s aberta há %.0fs — aguardando SL/TP.", pos.direction, age)
        return

    # ── 2. Busca candles ────────────────────────────────────────────────────
    logger.info("Buscando %s/%s…", tcfg.coin, tcfg.interval)
    df = fetch_candles(
        coin=tcfg.coin,
        interval=tcfg.interval,
        lookback=kcfg.lookback,
        base_url=cfg.hyperliquid.base_url,
    )
    logger.info("Close atual: %.4f", df["close"].iloc[-1])

    # ── 3. Sinal com ensemble + confirmação multi-TF ────────────────────────
    logger.info("Gerando sinal (sample_count=%d)…", kcfg.sample_count)
    signal = ensemble.get_signal(df, kcfg.pred_len, tcfg.interval)
    logger.info("Sinal: %s | conf=%.2f | %s",
                signal.direction.value, signal.confidence, signal.reason)

    monitor.alert_signal(tcfg.coin, signal.direction.value,
                         signal.confidence, signal.reason)

    if signal.direction == Direction.FLAT:
        logger.info("Sinal FLAT — ciclo encerrado sem ordem.")
        return

    # ── 4. Risk management ──────────────────────────────────────────────────
    balance = get_account_balance(cfg.hyperliquid) if not tcfg.dry_run else 1000.0
    order_params: OrderParams = calculate_order(signal, balance, tcfg)

    if order_params is None:
        logger.info("Risk manager bloqueou a ordem (balanço insuficiente ou FLAT).")
        return

    # ── 5. Envia ordem ──────────────────────────────────────────────────────
    result = place_order(order_params, cfg.hyperliquid, tcfg)

    monitor.alert_order_placed(
        tcfg.coin, signal.direction.value,
        order_params.size_usd, order_params.sl_price,
        order_params.tp_price, tcfg.dry_run,
    )

    # ── 6. Persiste estado ──────────────────────────────────────────────────
    state.set_position(OpenPosition(
        coin=tcfg.coin,
        direction=signal.direction.value,
        entry_price=signal.entry_price,
        size_usd=order_params.size_usd,
        sl_price=order_params.sl_price,
        tp_price=order_params.tp_price,
        opened_at=time.time(),
    ))
    logger.info("Estado de posição salvo.")


def main():
    cfg = Config()

    # ── Configuração ─────────────────────────────────────────────────────────
    cfg.trading.coin          = "BTC"
    cfg.trading.interval      = "1h"
    cfg.trading.dry_run       = True      # ⚠️ mude para False apenas após backtest
    cfg.trading.leverage      = 3
    cfg.trading.risk_per_trade_pct = 1.0
    cfg.trading.sl_pct        = 1.5
    cfg.trading.tp_pct        = 3.0
    cfg.trading.loop_interval_seconds = 3600

    cfg.kronos.lookback       = 200
    cfg.kronos.pred_len       = 10
    cfg.kronos.sample_count   = 10       # aumentar = mais lento, mais confiável

    # Para usar modelo fine-tunado (após rodar fine-tune):
    # cfg.kronos.model_name     = "finetune_csv/finetuned/HL_BTC_1h/basemodel/best_model"
    # cfg.kronos.tokenizer_name = "finetune_csv/finetuned/HL_BTC_1h/tokenizer/best_model"
    # ─────────────────────────────────────────────────────────────────────────

    ensemble = build_ensemble(cfg)
    state    = StateManager()
    cycle    = 0

    logger.info("Loop iniciado | %s/%s | dry_run=%s | sample_count=%d",
                cfg.trading.coin, cfg.trading.interval,
                cfg.trading.dry_run, cfg.kronos.sample_count)

    while True:
        cycle += 1
        try:
            run_cycle(ensemble, state, cfg, cycle)
        except Exception as exc:
            err = traceback.format_exc()
            logger.error("Erro no ciclo %d: %s", cycle, exc)
            monitor.alert_error(f"ciclo #{cycle}", err)

        logger.info("Próximo ciclo em %ds…", cfg.trading.loop_interval_seconds)
        time.sleep(cfg.trading.loop_interval_seconds)


if __name__ == "__main__":
    main()
""",
}

for path, content in CUSTOM_FILES.items():
    os.makedirs(os.path.dirname('/content/Kronos/' + path), exist_ok=True)
    with open('/content/Kronos/' + path, 'w') as f:
        f.write(content)

print('✅ Arquivos custom escritos:', list(CUSTOM_FILES.keys()))


## 3. Instala dependências

In [ ]:
!pip install -q -r requirements.txt
!pip install -q ccxt

# Verifica GPU
import torch
print(f'GPU disponível: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Dispositivo: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 4. Opção A — Baixa dados direto da Binance (recomendado)
Não precisa de upload manual. Faz o download aqui mesmo.

In [ ]:
# ── Configure aqui ─────────────────────────────────────────────────────
SYMBOL   = 'BTC'    # BTC | ETH | SOL
INTERVAL = '10m'  # 10m | 1h | 4h
DAYS     = 730    # 2 anos de 10m ≈ 105k candles
# ───────────────────────────────────────────────────────────────────────

import sys
sys.path.insert(0, '/content/Kronos')
from infra.binance_data_pipeline import build_exchange, fetch_symbol_history, save_csv

exchange = build_exchange()
df = fetch_symbol_history(exchange, SYMBOL, INTERVAL, DAYS)
csv_path = save_csv(df, SYMBOL, INTERVAL)

# Copia para o Drive (persistência entre sessões)
import shutil
drive_csv = f'{DRIVE_ROOT}/data/BN_{SYMBOL}_{INTERVAL}.csv'
shutil.copy(csv_path, drive_csv)
print(f'✅ Backup no Drive: {drive_csv}')

## 4.2 Baixa dados da Hyperliquid

A Hyperliquid tem histórico desde ~Out/2025. Este passo baixa os dados da exchange-alvo
que são essenciais para a domain adaptation (candles do mercado exato onde vamos operar).

In [ ]:
import subprocess, sys, os, shutil

SYMBOL   = 'BTC'   # deve bater com o da célula 4
INTERVAL = '10m'  # deve bater com a célula acima
# HL não tem 10m nativo — baixamos 5m e o build_training_dataset resampla
HL_INTERVAL = '5m' if INTERVAL == '10m' else INTERVAL
HL_DAYS  = 185

os.chdir('/content/Kronos')
# save_dataset salva em finetune_csv/data/HL_{coin}_{interval}.csv automaticamente
hl_local = f'/content/Kronos/finetune_csv/data/HL_{SYMBOL}_{HL_INTERVAL}.csv'
hl_drive = f'{DRIVE_ROOT}/data/HL_{SYMBOL}_{HL_INTERVAL}.csv'

if os.path.exists(hl_drive):
    os.makedirs('/content/Kronos/finetune_csv/data', exist_ok=True)
    shutil.copy(hl_drive, hl_local)
    print(f'Copiado do Drive: {hl_local}')
else:
    print('Baixando dados da Hyperliquid...')
    result = subprocess.run(
        [sys.executable, '-m', 'infra.crypto_data_pipeline',
         '--coin', SYMBOL, '--interval', HL_INTERVAL, '--days', str(HL_DAYS)],
        capture_output=True, text=True
    )
    print(result.stdout)
    if result.returncode != 0:
        print('ERRO:', result.stderr)
        raise RuntimeError('crypto_data_pipeline falhou')
    os.makedirs(f'{DRIVE_ROOT}/data', exist_ok=True)
    shutil.copy(hl_local, hl_drive)
    print(f'Salvo no Drive: {hl_drive}')

import pandas as pd
df_hl = pd.read_csv(hl_local)
print(f'HL: {len(df_hl):,} candles | {df_hl["timestamps"].min()} → {df_hl["timestamps"].max()}')


## 4. Opção B — Upload manual do CSV gerado localmente
Use se já rodou o pipeline localmente.

In [ ]:
# from google.colab import files
# uploaded = files.upload()  # selecione BN_BTC_1h.csv
#
# import shutil
# for fname in uploaded:
#     shutil.move(fname, f'/content/Kronos/finetune_csv/data/{fname}')
#     print(f'Movido: {fname}')

## 4.5 Combina Binance + Hyperliquid (domain adaptation)

Gera `COMBINED_BTC_1h.csv` mesclando o histórico longo da Binance (padrões gerais) com
o histórico da Hyperliquid (distribuição exata do alvo):
- Candles Binance **anteriores** ao início do dataset HL → contexto de pré-treino
- Candles Hyperliquid → período recente, preços exatos da exchange-alvo

O script também gera `config_combined_btc_1h.yaml` automaticamente.

In [ ]:
import subprocess, sys, os

SYMBOL   = 'BTC'   # deve bater com o da célula 4
INTERVAL = '10m'

os.chdir('/content/Kronos')

result = subprocess.run(
    [sys.executable, '-m', 'infra.build_training_dataset',
     '--symbol', SYMBOL, '--interval', INTERVAL],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print('ERRO:', result.stderr)
    raise RuntimeError('build_training_dataset falhou')

# Verifica o CSV gerado
import pandas as pd
combined_path = f'finetune_csv/data/COMBINED_{SYMBOL}_{INTERVAL}.csv'
df = pd.read_csv(combined_path)
print(f'\nCombined dataset: {len(df):,} candles | colunas: {list(df.columns)}')
print(df[['timestamps','open','close','volume','amount']].head(3))
print('...')
print(df[['timestamps','open','close','volume','amount']].tail(3))

# Salva no Drive para reutilizar em sessões futuras
import shutil, os
os.makedirs(f'{DRIVE_ROOT}/data', exist_ok=True)
shutil.copy(combined_path, f'{DRIVE_ROOT}/data/COMBINED_{SYMBOL}_{INTERVAL}.csv')
print(f'✅ Dataset combinado salvo no Drive: {DRIVE_ROOT}/data/COMBINED_{SYMBOL}_{INTERVAL}.csv')

## 5. Gera config de fine-tune e ajusta paths para o Colab

In [ ]:
import yaml, os

SYMBOL   = 'BTC'   # deve bater com o da célula 4
INTERVAL = '10m'

config = {
    'data': {
        'data_path':       f'/content/Kronos/finetune_csv/data/COMBINED_{SYMBOL}_{INTERVAL}.csv',
        'lookback_window': 512,
        'predict_window':  6,   # 6 × 10min = 60 min de horizonte
        'max_context':     512,
        'clip':            5.0,
        'train_ratio':     0.85,
        'val_ratio':       0.10,
        'test_ratio':      0.05,
    },
    'training': {
        'tokenizer_epochs':        30,
        'basemodel_epochs':        20,
        'batch_size':              32,
        'log_interval':            50,
        'num_workers':             2,
        'seed':                    42,
        'tokenizer_learning_rate': 2e-4,
        'predictor_learning_rate': 1e-6,
        'adam_beta1':              0.9,
        'adam_beta2':              0.95,
        'adam_weight_decay':       0.1,
        'accumulation_steps':      1,
    },
    'model_paths': {
        'pretrained_tokenizer': 'NeoQuasar/Kronos-Tokenizer-base',
        'pretrained_predictor': 'NeoQuasar/Kronos-small',
        'exp_name':             f'COMBINED_{SYMBOL}_{INTERVAL}',
        # Salva direto no Drive — sobrevive ao fim da sessão
        'base_path':            f'{DRIVE_ROOT}/finetuned',
        'base_save_path':       '',
        'finetuned_tokenizer':  '',
        'tokenizer_save_name':  'tokenizer',
        'basemodel_save_name':  'basemodel',
    },
    'experiment': {
        'name':                f'kronos_combined_{SYMBOL.lower()}_{INTERVAL}',
        'description':         f'Kronos fine-tuned on Binance+Hyperliquid combined {SYMBOL} {INTERVAL} (domain adaptation, Colab)',
        'use_comet':           False,
        'train_tokenizer':     True,
        'train_basemodel':     True,
        'skip_existing':       False,
        'pre_trained_tokenizer': True,
        'pre_trained_predictor': True,
    },
    'device': {'use_cuda': True, 'device_id': 0},
}

cfg_path = f'/content/Kronos/finetune_csv/configs/config_combined_{SYMBOL.lower()}_{INTERVAL}.yaml'
with open(cfg_path, 'w') as f:
    yaml.dump(config, f, default_flow_style=False)

print(f'✅ Config salvo em: {cfg_path}')
print(f'   Checkpoints vão para: {DRIVE_ROOT}/finetuned/COMBINED_{SYMBOL}_{INTERVAL}/')

## 6. Treina o Kronos 🚀

In [ ]:
import subprocess, sys

SYMBOL   = 'BTC'
INTERVAL = '10m'
cfg_path = f'/content/Kronos/finetune_csv/configs/config_combined_{SYMBOL.lower()}_{INTERVAL}.yaml'

cmd = [
    sys.executable,
    '/content/Kronos/finetune_csv/train_sequential.py',
    '--config', cfg_path
]

print('Iniciando treinamento...')
print(f'Comando: {" ".join(cmd)}\n')

import os as _os
_env = _os.environ.copy()
_env['PYTHONPATH'] = '/content/Kronos'
result = subprocess.run(cmd, cwd='/content/Kronos', env=_env)

if result.returncode == 0:
    print('\n✅ Treinamento concluído!')
    print(f'Modelo salvo em: {DRIVE_ROOT}/finetuned/COMBINED_{SYMBOL}_{INTERVAL}/')
else:
    print('\n❌ Erro no treinamento — veja o output acima')

## 7. Valida o modelo treinado

In [ ]:
import sys, pandas as pd
sys.path.insert(0, '/content/Kronos')

from model import Kronos, KronosTokenizer, KronosPredictor

SYMBOL   = 'BTC'
INTERVAL = '1h'
tok_path  = f'{DRIVE_ROOT}/finetuned/BN_{SYMBOL}_{INTERVAL}/tokenizer/best_model'
pred_path = f'{DRIVE_ROOT}/finetuned/BN_{SYMBOL}_{INTERVAL}/basemodel/best_model'

tokenizer = KronosTokenizer.from_pretrained(tok_path)
model     = Kronos.from_pretrained(pred_path)
predictor = KronosPredictor(model, tokenizer, device='cuda', max_context=512)

# Testa com os últimos 200 candles do CSV
df  = pd.read_csv(f'/content/Kronos/finetune_csv/data/BN_{SYMBOL}_{INTERVAL}.csv')
df['timestamps'] = pd.to_datetime(df['timestamps'])

x_df = df.tail(200)[['open','high','low','close','volume']].reset_index(drop=True)
x_ts = df.tail(200)['timestamps'].reset_index(drop=True)

from datetime import timedelta
y_ts = pd.Series([x_ts.iloc[-1] + timedelta(hours=i+1) for i in range(24)])

forecast = predictor.predict(df=x_df, x_timestamp=x_ts, y_timestamp=y_ts,
                             pred_len=24, T=1.0, top_p=0.9, sample_count=3)

print(f'Close atual : ${df["close"].iloc[-1]:,.2f}')
print(f'Forecast +24h: ${forecast["close"].iloc[-1]:,.2f}')
print(forecast[['close','high','low']].head(6))